# FRED Stage 2: Model Testing & Comprehensive Benchmark Suite
### Automated Evaluation across Clean Preservation, Complete Edit Success & Selective Actions

This notebook evaluates your fine-tuned **Qwen2.5-3B** and **Qwen2.5-7B** models against the official Stage 2 evaluation benchmarks:
- **`test_clean.jsonl`**: Untouched official FinQA and TAT-QA test sets to evaluate **Clean Preservation Rate (CPR)** and **False Correction Rate (FCR)**.
- **`test_corrupted.jsonl`**: Controlled hallucination test benchmark across all 11 error types to evaluate **Complete Edit Success (CES)** and **Action Macro F1**.
- **Commercial LLM Comparator**: Includes a dedicated prompt extraction cell to test identical samples on **ChatGPT (GPT-4o)**, **Claude 3.5 Sonnet**, and **Gemini 1.5 Pro**.

## 1. Setup & Environment

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" peft accelerate bitsandbytes datasets tqdm

## 2. Load Fine-Tuned Model from Hugging Face Hub (or Local Checkpoint)

In [ ]:
import torch
from unsloth import FastLanguageModel
import json
import re
from tqdm import tqdm

# ── Model Source Configuration ───────────────────────────────────────────────
# Enter your Hugging Face adapter ID (e.g., 'your_username/fred-stage2-qwen2.5-3b-lora')
# or local path if testing in the same Kaggle session (e.g., 'fred_stage2_qwen2_5_3b_lora')
LORA_MODEL_ID = input("Enter Hugging Face LoRA repo ID or local path: ").strip() or "unsloth/Qwen2.5-3B-Instruct"

MAX_SEQ_LENGTH = 2048
print(f"Loading model: {LORA_MODEL_ID}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model loaded for inference!")

## 3. Load Test Benchmark Datasets
Loads `test_clean.jsonl` and `test_corrupted.jsonl`.

In [ ]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

clean_test_set = load_jsonl("test_clean.jsonl")
corrupted_test_set = load_jsonl("test_corrupted.jsonl")

print(f"Loaded {len(clean_test_set):,} clean test records (Target: Clean Preservation)")
print(f"Loaded {len(corrupted_test_set):,} corrupted test records (Target: Editing & Selective Actions)")

# Test slice size (set to None to evaluate all 1,200 records, or e.g. 100 for fast eval)
EVAL_LIMIT = 100

## 4. Run Inference Pipeline

In [ ]:
def run_prediction(context, table, question, candidate_response):
    table_str = "\n".join([" | ".join(r) for r in table]) if table else "[No Table]"
    user_content = (
        f"Context: {context}\n\n"
        f"Table:\n{table_str}\n\n"
        f"Question: {question}\n\n"
        f"Response to Verify: {candidate_response}"
    )
    messages = [
        {
            "role": "system",
            "content": (
                "You are FRED Stage 2, an evidence-aware financial hallucination verifier. "
                "Given context, tables, a question, and a response, determine the appropriate action: "
                "preserve, edit, abstain, flag_conflict, or request_evidence."
            )
        },
        {"role": "user", "content": user_content}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    raw_output = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    
    # Parse action from JSON payload or fallback regex
    action = "preserve"
    try:
        parsed = json.loads(raw_output)
        action = parsed.get("action", "preserve")
    except Exception:
        m = re.search(r'"action"\s*:\s*"(preserve|edit|abstain|flag_conflict|request_evidence)"', raw_output, re.I)
        if m:
            action = m.group(1).lower()
        elif "<delete>" in raw_output:
            action = "edit"
        elif "unverifiable" in raw_output.lower():
            action = "abstain"
    return action, raw_output

## 5. Evaluate Clean Preservation Rate (CPR) & False Correction Rate (FCR)
Runs inference on untouched official clean test data.

In [ ]:
print("\n--- 1. Evaluating Clean Preservation on test_clean.jsonl ---")
clean_subset = clean_test_set[:EVAL_LIMIT] if EVAL_LIMIT else clean_test_set

clean_preserved = 0
clean_falsely_edited = 0
clean_abstentions = 0

for r in tqdm(clean_subset, desc="Evaluating Clean Data"):
    action, _ = run_prediction(
        context=r.get("context", ""),
        table=r.get("table", []),
        question=r.get("question", ""),
        candidate_response=r.get("unverified_response", r.get("gold_answer", ""))
    )
    if action == "preserve":
        clean_preserved += 1
    elif action == "edit":
        clean_falsely_edited += 1
    elif action in ["abstain", "request_evidence"]:
        clean_abstentions += 1

cpr = (clean_preserved / len(clean_subset)) * 100.0
fcr = (clean_falsely_edited / len(clean_subset)) * 100.0

print(f"\nClean Preservation Results (Total: {len(clean_subset)}):")
print(f"  [CPR] Clean Preservation Rate: {cpr:.1f}%  (Target: >= 85.0% | Stage 1 Baseline: 40.5%)")
print(f"  [FCR] False Correction Rate:   {fcr:.1f}%  (Target: <= 10.0% | Stage 1 Baseline: 59.5%)")

## 6. Evaluate Hallucination Detection & Complete Edit Success (CES)
Runs inference on corrupted test benchmark suite.

In [ ]:
print("\n--- 2. Evaluating Hallucination Detection & Editing on test_corrupted.jsonl ---")
corrupt_subset = corrupted_test_set[:EVAL_LIMIT] if EVAL_LIMIT else corrupted_test_set

action_correct = 0
edit_attempts = 0
edit_ces = 0

for r in tqdm(corrupt_subset, desc="Evaluating Corrupted Data"):
    target_action = r.get("correct_action", r.get("action", "edit"))
    action, raw_resp = run_prediction(
        context=r.get("context", ""),
        table=r.get("table", []),
        question=r.get("question", ""),
        candidate_response=r.get("modified_answer", r.get("unverified_response", ""))
    )
    if action == target_action:
        action_correct += 1
        
    if target_action == "edit":
        edit_attempts += 1
        # Check if output contains valid XML tags
        if "<delete>" in raw_resp and "<mark>" in raw_resp:
            edit_ces += 1

action_accuracy = (action_correct / len(corrupt_subset)) * 100.0
ces_rate = (edit_ces / edit_attempts * 100.0) if edit_attempts > 0 else 0.0

print(f"\nCorrupted Benchmark Results (Total: {len(corrupt_subset)}):")
print(f"  Selective Action Accuracy:     {action_accuracy:.1f}%")
print(f"  Complete Edit Success (CES):   {ces_rate:.1f}%  (Stage 1 Baseline: 46.9%)")

--- 
## 7. Commercial Model Comparison Generator (GPT-4o, Claude 3.5, Gemini 1.5)
Generates clean copy-paste prompts for small-scale manual chat testing in web/mobile apps.

In [ ]:
print("Generating 5 diverse test samples for manual commercial model testing...\n")
sample_prompts = []

for idx in range(min(5, len(corrupted_test_set))):
    sample = corrupted_test_set[idx]
    t_str = "\n".join([" | ".join(r) for r in sample.get("table", [])]) if sample.get("table") else "[No Table]"
    prompt_text = f"""============================================================
TEST SAMPLE #{idx+1} [Target Action: {sample.get('correct_action', sample.get('action'))}]
============================================================
You are FRED Stage 2, a financial hallucination verifier.
Determine the action: preserve, edit, abstain, flag_conflict, or request_evidence.
Respond in JSON.

Context: {sample.get('context')}

Table:
{t_str}

Question: {sample.get('question')}

Response to Verify: {sample.get('modified_answer', sample.get('unverified_response'))}
"""
    print(prompt_text)
    sample_prompts.append(prompt_text)